In [ ]:
%load_ext autoreload
%autoreload 2

from typing import Final

import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import welch

from src.utils_jr import simulate_jr, alpha_delta_spectral_power, JRParams

def plot_signal(
        _t: np.ndarray, 
        _eeg: np.ndarray,
        _power_times: np.ndarray,
        _alpha_power: np.ndarray,
        _delta_power: np.ndarray) -> None:
    _, axes = plt.subplots(
        3, 1, figsize=(12, 10), sharex=True
    )

    axes[0].plot(_t, _eeg, linewidth=0.7)
    axes[0].set_ylabel("JR output (mV)")
    axes[0].set_title("Jansen-Rit output")

    axes[1].plot(
        _power_times, _alpha_power, marker="o", markersize=3
    )
    axes[1].set_xlabel("Time (s)")
    axes[1].set_ylabel(r"Alpha power (mV$^2$)")
    axes[1].set_title(f"Alpha spectral power; median {round(np.median(_alpha_power), 3)}")

    axes[2].plot(_power_times, _delta_power)
    axes[2].set_ylabel(r"Delta power (mV$^2$")
    axes[2].set_title(f"Delta spectral power; median {round(np.median(_delta_power), 3)}")

    for ax in axes:
        ax.grid(True, alpha=.3)

    plt.tight_layout()
    plt.show()

sf: Final[float] = 500. # sampling frequency
for B_value in (20, 21, 22, 25, 29, 30, 31, 32, 33, 34):
    t, eeg, states = simulate_jr(
        duration=20.0,
        fs=sf,
        external_rate= 120,
        params=JRParams(
            A=3.9,
            B=B_value # 21 to 29: 29 - maximum Delta, 21 - maximum Alpha
            )
    )

    power_times, alpha_power, delta_power = alpha_delta_spectral_power(
        t,
        eeg,
        sfreq=sf,
        window_sec=4.0,
        step_sec=1.0,
    )

    freqs, psd = welch(eeg, fs=sf, nperseg=2500)
    plt.figure()
    plt.plot(freqs, psd)
    plt.xlim(0,20)
    plt.title(f"PSD for B={B_value}")
    plt.show()


    plot_signal(
        _t=t,
        _eeg=eeg,
        _power_times = power_times,
        _alpha_power=alpha_power,
        _delta_power=delta_power
    )

In [ ]:
def median_alpha_delta(ext_rate: float = 120, A: float = 3.25, B: float = 22.) -> tuple[float, float]:
    t, eeg, _ = simulate_jr(duration=20., fs=500.,
                            external_rate=ext_rate,
                            params=JRParams(A=A, B=B))
    _, alpha_pow, delta_pow = alpha_delta_spectral_power(t, eeg, sfreq=500., window_sec=4., step_sec=1.)
    return float(np.median(alpha_pow)), float(np.median(delta_pow))

median_alpha_delta()

In [ ]:
from itertools import product
from tqdm import tqdm


def grid_search_jr(
    A_values: np.ndarray,
    B_values: np.ndarray,
    ext_rate: float = 120.0,
    min_alpha_pow: float = 1e-4,
    min_delta_pow: float = 1e-4,
) -> list[dict]:
    results = []

    for A, B in tqdm(list(product(A_values, B_values))):
        alpha_pow, delta_pow = median_alpha_delta(
            ext_rate=ext_rate,
            A=float(A),
            B=float(B),
        )

        # Reject trials with insufficient absolute power.
        if (
            not np.isfinite(alpha_pow)
            or not np.isfinite(delta_pow)
            or alpha_pow < min_alpha_pow
            or delta_pow < min_delta_pow
        ):
            continue

        score = delta_pow / alpha_pow

        result = {
            "A": float(A),
            "B": float(B),
            "alpha_pow": alpha_pow,
            "delta_pow": delta_pow,
            "ratio": score,
        }
        results.append(result)

    results.sort(key=lambda x: x["ratio"], reverse=True)
    return results


A_values = np.linspace(2.5, 4.0, 16)
B_values = np.linspace(15.0, 30.0, 16)

results = grid_search_jr(
    A_values=A_values,
    B_values=B_values,
    ext_rate=120.0,
    min_alpha_pow=1e-4,
    min_delta_pow=1e-4,
)

print("\nTop 10 valid trials:")
for result in results[:10]:
    print(result)

In [ ]:
from src.utils_jr import simulate_jr, alpha_delta_spectral_power, JRParams
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import welch
from typing import Final

alpha_band: Final[tuple[float, float]] = (8.0, 13.0)
delta_band: Final[tuple[float, float]] = (0.5, 4.0)

B_values = np.arange(15, 36, 1)

# Containers
median_alpha = []
median_delta = []
psds = []          # list of 1D arrays, one per B
freqs_ref = None   # shared frequency axis

for B_value in B_values:
    t, eeg, states = simulate_jr(
        duration=20.0,
        fs=sf,
        external_rate=120,
        params=JRParams(
            A=3.9,
            B=float(B_value),
        ),
    )

    power_times, alpha_power, delta_power = alpha_delta_spectral_power(
        t,
        eeg,
        sfreq=sf,
        window_sec=4.0,
        step_sec=1.0,
    )

    freqs, psd = welch(eeg, fs=sf, nperseg=2500)

    median_alpha.append(np.median(alpha_power))
    median_delta.append(np.median(delta_power))

    if freqs_ref is None:
        freqs_ref = freqs
    psds.append(psd)

median_alpha = np.asarray(median_alpha)
median_delta = np.asarray(median_delta)
psd_2d = np.vstack(psds)  # shape (n_B, n_freqs)

Median alpha & delta power vs. B

In [ ]:
fig, ax = plt.subplots()
ax.plot(B_values, median_alpha, "o-", label=f"median alpha power ({alpha_band[0]}–{alpha_band[1]} Hz)")
ax.plot(B_values, median_delta, "s-", label=f"median delta power ({alpha_band[0]}–{alpha_band[1]} Hz)")
ax.set_xlabel("B")
ax.set_ylabel("median spectral power")
ax.set_title(r"Median $\alpha$ and $\Delta$ power vs. B")
ax.legend()
ax.grid(True)
plt.show()

PSD surface: frequency (0–20 Hz) vs. B

In [ ]:
mask = (freqs_ref >= 0) & (freqs_ref <= 30)
freqs_lim = freqs_ref[mask]
psd_lim = psd_2d[:, mask]

F, Bgrid = np.meshgrid(freqs_lim, B_values)

fig = plt.figure()
ax3d = fig.add_subplot(111, projection="3d")
surf = ax3d.plot_surface(F, Bgrid, psd_lim, cmap="viridis", linewidth=0, antialiased=True)
ax3d.set_xlabel("Frequency (Hz)")
ax3d.set_ylabel("B")
# ax3d.set_zlabel("PSD")
ax3d.set_title("PSD vs. frequency and B")
fig.colorbar(surf, ax=ax3d, shrink=0.6, label="PSD")
plt.show()

In [ ]:
fig, ax = plt.subplots()
im = ax.imshow(
    psd_lim,
    aspect="auto",
    origin="lower",
    extent=[freqs_lim[0], freqs_lim[-1], B_values[0], B_values[-1]],
    cmap="viridis",
)
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("B")
ax.set_title("PSD heatmap (0–30 Hz)")
fig.colorbar(im, ax=ax, label="PSD")
plt.show()

In [ ]:
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.colors import LogNorm

# Band definitions (boundaries in Hz)
bands = [
    ("delta", delta_band[0], delta_band[1],  "#1f77b4"),
    ("theta", delta_band[1], alpha_band[0],  "#2ca02c"),
    ("alpha", alpha_band[0], alpha_band[1], "#ff7f0e"),
    ("beta", alpha_band[1], 30, "#e96c7f"),
]

for log_norm in False, True:
    fig, ax = plt.subplots(figsize=(8, 5))

    im = ax.imshow(
        psd_lim,
        aspect="auto",
        origin="lower",
        extent=[freqs_lim[0], freqs_lim[-1], B_values[0], B_values[-1]],
        cmap="viridis",
        norm=LogNorm(vmin=psd_lim.min(), vmax=psd_lim.max()) if log_norm else None,
    )
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("B")
    # ax.set_title("PSD heatmap (0–20 Hz) with band boundaries")
    ax.set_xlim(freqs_lim[0], freqs_lim[-1])
    fig.colorbar(im, ax=ax, label="PSD")

    # --- Attach a slim axes above the heatmap for the band track ---
    divider = make_axes_locatable(ax)
    band_ax = divider.append_axes("top", size="7%", pad=0.05, sharex=ax)

    # Draw a horizontal band strip for each frequency band
    for name, lo, hi, color in bands:
        band_ax.axvspan(lo, hi, color=color, alpha=0.6)
        band_ax.text(
            (lo + hi) / 2, 0.5, name,
            ha="center", va="center",
            fontsize=9, color="black", transform=band_ax.get_xaxis_transform(),
        )

    # Vertical boundary lines aligned with the heatmap
    for _, lo, hi, _ in bands:
        for edge in (lo, hi):
            ax.axvline(edge, color="white", lw=0.8, ls="--", alpha=0.8)
            band_ax.axvline(edge, color="black", lw=0.8, ls="--", alpha=0.6)

    # Cosmetics: strip is just a legend, hide ticks/labels
    band_ax.set_ylim(0, 1)
    band_ax.set_yticks([])
    # band_ax.set_xticks([])
    band_ax.tick_params(axis="x", which="both", bottom=False, top=False, labelbottom=False)
    
    title = "PSD heatmap (0-30 Hz)"
    if log_norm:
        title += " in log scale"

    band_ax.set_title(title, fontsize=12, pad=5)
    # fig.suptitle("PSD heatmap (0–20 Hz)", fontsize=13)

    plt.show()